# 02 — Tool calling: el LLM decide qué herramienta usar

**Level 3 — Agentic AI & Workflows**

Ollama tiene tool calling nativo: le declaramos herramientas disponibles,
el modelo decide cuál ejecutar (con sus argumentos), y nosotros la
ejecutamos y devolvemos el resultado. El notebook muestra esa decisión
paso a paso.

In [1]:
import json
import os

import requests
from dotenv import load_dotenv

load_dotenv()

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MODELO = "llama3.2"


def get_hora() -> str:
    """Tool 1: devuelve la hora actual."""
    from datetime import datetime
    return datetime.now().strftime("%H:%M:%S")


def suma(a: int, b: int) -> int:
    """Tool 2: suma dos numeros enteros."""
    return a + b

## Declarar las herramientas (JSON)

In [2]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_hora",
            "description": "Devuelve la hora actual en formato HH:MM:SS",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "suma",
            "description": "Suma dos numeros enteros",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"},
                },
                "required": ["a", "b"],
            },
        },
    },
]

EJECUTAR = {
    "get_hora": lambda args: get_hora(),
    "suma": lambda args: suma(args["a"], args["b"]),
}

## Pregunta 1: ¿Qué hora es?

In [3]:
pregunta = "Que hora es?"
payload = {
    "model": MODELO,
    "messages": [{"role": "user", "content": pregunta}],
    "tools": TOOLS,
    "stream": False,
}
response = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload, timeout=120)
response.raise_for_status()
mensaje = response.json()["message"]

print(f"Contenido del modelo: {mensaje.get('content', '')!r}")
print(f"Tool calls pedidos: {mensaje.get('tool_calls')}")

for llamada in mensaje.get("tool_calls", []):
    nombre = llamada["function"]["name"]
    argumentos = llamada["function"]["arguments"]
    if isinstance(argumentos, str):
        argumentos = json.loads(argumentos or "{}")
    resultado = EJECUTAR[nombre](argumentos)
    print(f"-> Ejecutando {nombre}{argumentos}: {resultado}")

Contenido del modelo: ''
Tool calls pedidos: [{'id': 'call_9ld26o3q', 'function': {'index': 0, 'name': 'get_hora', 'arguments': {}}}]
-> Ejecutando get_hora{}: 13:52:51


## Pregunta 2: una suma

In [4]:
pregunta = "Cuanto es 17 + 25?"
payload = {
    "model": MODELO,
    "messages": [{"role": "user", "content": pregunta}],
    "tools": TOOLS,
    "stream": False,
}
response = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload, timeout=120)
response.raise_for_status()
mensaje = response.json()["message"]

print(f"Contenido del modelo: {mensaje.get('content', '')!r}")
print(f"Tool calls pedidos: {mensaje.get('tool_calls')}")

for llamada in mensaje.get("tool_calls", []):
    nombre = llamada["function"]["name"]
    argumentos = llamada["function"]["arguments"]
    if isinstance(argumentos, str):
        argumentos = json.loads(argumentos or "{}")
    resultado = EJECUTAR[nombre](argumentos)
    print(f"-> Ejecutando {nombre}{argumentos}: {resultado}")

Contenido del modelo: ''
Tool calls pedidos: [{'id': 'call_vdjkovbi', 'function': {'index': 0, 'name': 'suma', 'arguments': {'a': '17', 'b': '25'}}}]
-> Ejecutando suma{'a': '17', 'b': '25'}: 1725


## Conclusión

- El modelo **pidió ejecutar** `get_hora` (no inventó la hora)
- Para la suma pidió `suma` con `a=17, b=25` → 42
- El `content` queda vacío: el modelo solo pidió la tool, no generó texto
- Esa decisión — no el texto — es el inicio de los agentes